In [ ]:
import re
from pathlib import Path
import pandas as pd

In [ ]:
# File paths
input_csv = Path("data/raw/scopus_export.csv")
cleaned_csv = Path("data/processed/scopus_cleaned.csv")
formatted_csv = Path("data/processed/scopus_bibliometrix_formatted.csv")

In [ ]:
def clean_text(text):
    """Remove HTML tags from text."""
    return re.sub(r"<.*?>", "", str(text))

def clean_scopus_data(input_path, output_path):
    """Clean Scopus-exported bibliographic data."""

    df = pd.read_csv(input_path, encoding="utf-8")

    # Remove duplicate records
    df = df.drop_duplicates(subset=["Title", "DOI"], keep="first")

    # Remove incomplete records
    df = df.dropna(subset=["Title", "DOI", "Author Keywords", "Index Keywords"])

    # Clean HTML tags
    df = df.map(clean_text) if hasattr(df, "map") else df.applymap(
        lambda x: clean_text(x) if isinstance(x, str) else x)

    df.to_csv(output_path, index=False)

    print(f"Cleaned dataset exported to {output_path}. Total records: {len(df)}")

# Execute cleaning
clean_scopus_data(input_csv, cleaned_csv)

In [ ]:
# Load cleaned dataset
df = pd.read_csv(cleaned_csv)

# Add required empty columns if missing
for col in ["JI", "WC"]:
    if col not in df.columns:
        df[col] = ""


# Column mapping for Bibliometrix/Biblioshiny
column_mapping = {
    "Authors": "AU",
    "Title": "TI",
    "DOI": "DI",
    "Source title": "SO",
    "Year": "PY",
    "Cited by": "TC",
    "Affiliations": "C1",
    "Author Keywords": "DE",
    "Index Keywords": "ID",
    "Abstract": "AB",
    "Correspondence Address": "RP",
    "References": "CR",}

df.rename(columns={col: column_mapping[col] for col in df.columns if col in column_mapping},
    inplace=True,)

df.to_csv(formatted_csv, index=False)

print(f"Formatted dataset exported to {formatted_csv}")